#### Gaussian Process fitting behavioral responses

In [ ]:
# All Imports
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

In [ ]:
from config import dir_config, main_config

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)

# Following viewing times to be added in reaction time to account for actual accumulation time
min_viewing_time_uw = 0.3
min_viewing_time_ucla = 0.3

acceptance_criteria = main_config.acceptance_criteria

In [1]:
import GPy

# Remove NaN rows from all_data to make valid_data
valid_data = all_data.dropna(subset=["signed_coherence", "choice"])

# Normalize coherence values to 0–1 range
coherence = np.array(valid_data["signed_coherence"]).reshape(-1, 1)
coherence = (coherence - coherence.min()) / (coherence.max() - coherence.min())

# Compute aggregated psychometric data
unique_coherence, responses = get_psych(valid_data)

# Remove NaNs (if any) from psychometric data
valid_indices = ~np.isnan(responses)
unique_coherence = unique_coherence[valid_indices].reshape(-1, 1)
responses = responses[valid_indices].reshape(-1, 1)

# Normalize unique coherence values to 0–1 for regression
unique_coherence = (unique_coherence - unique_coherence.min()) / (unique_coherence.max() - unique_coherence.min())

# Define RBF kernel
kernel = GPy.kern.RBF(input_dim=1, lengthscale=0.1)

# Fit GP regression model
gp_model = GPy.models.GPRegression(unique_coherence, responses, kernel)
gp_model.optimize(messages=True)

# Predict probabilities for new coherence levels
new_coherence = np.linspace(0, 1, 100).reshape(-1, 1)
mean, variance = gp_model.predict(new_coherence)

# Plot
plt.figure()
plt.scatter(unique_coherence, responses, color="red", label="Observed Proportions")
plt.plot(new_coherence, mean, color="blue", label="GP Mean")
plt.fill_between(new_coherence.flatten(), (mean - 2 * np.sqrt(variance)).flatten(), (mean + 2 * np.sqrt(variance)).flatten(), color="blue", alpha=0.2, label="Confidence Interval")
plt.legend()
plt.xlabel("Coherence (Normalized)")
plt.ylabel("P(+1 Response)")
plt.title("Psychometric Function Modeled with GP Regression")
plt.show()

ModuleNotFoundError: No module named 'GPy'

In [2]:
import GPy
import matplotlib.pyplot as plt
import numpy as np

# Initial stimuli and responses
stimuli = np.array([-100, -72, -36, -18, -9, 0, 9, 18, 36, 72, 100]).reshape(-1, 1)
responses = np.array([+1, -1, +1, -1, +1, -1, -1, +1, -1, +1, +1])
responses = (responses + 1) / 2  # Convert +1/-1 to 1/0
responses = responses.reshape(-1, 1)

# Define initial GP kernel
kernel = GPy.kern.RBF(input_dim=1, lengthscale=20)


# Define acquisition function
def select_next_stimulus(gp_model, stimulus_range):
	new_stimuli = stimulus_range.reshape(-1, 1)
	_, variance = gp_model.predict(new_stimuli)
	next_stimulus_idx = np.argmax(variance)
	return new_stimuli[next_stimulus_idx]


# Training loop
stimulus_range = np.linspace(-100, 100, 200)  # Stimulus range
n_trials = 20  # Number of trials

for _trial in range(n_trials):
	# Step 1: Create GP model with current stimuli and responses
	try:
		gp_model = GPy.models.GPClassification(stimuli, responses, kernel=kernel.copy())
		gp_model.optimize(messages=False)
	except Exception as e:
		break

	# Step 2: Select next stimulus
	next_stimulus = select_next_stimulus(gp_model, stimulus_range)

	# Step 3: Simulate response (replace with actual experiment)
	response = np.random.choice([0, 1])  # Simulate response (0 or 1)

	# Step 4: Append new data
	stimuli = np.vstack([stimuli, next_stimulus])  # Add new stimulus
	responses = np.append(responses, response).reshape(-1, 1)  # Add new response and reshape

# After training, the GP should adapt to the stimuli-responses data.

ModuleNotFoundError: No module named 'GPy'